# Инициализация Spark и подключение к БД

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import broadcast


spark = SparkSession.builder \
    .appName("ETL_Lab2_Final_Fixed_Integrity") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2,com.clickhouse:clickhouse-jdbc:0.6.0") \
    .getOrCreate()


pg_url = "jdbc:postgresql://db:5432/pet_store_db"
pg_properties = {"user": "admin", "password": "password", "driver": "org.postgresql.Driver"}
ch_url = "jdbc:clickhouse://clickhouse:8123/reports_db?compress=false&ssl=false"
ch_properties = {"user": "admin", "password": "password", "driver": "com.clickhouse.jdbc.ClickHouseDriver"}

# Загрузка и предобработка исходных данных

In [2]:
raw_df = spark.read.jdbc(pg_url, "mock_data", properties=pg_properties)

for field in raw_df.schema.fields:
    if isinstance(field.dataType, StringType):
        raw_df = raw_df.withColumn(field.name, F.trim(F.col(field.name)))

raw_df = raw_df.cache()

# Заполнение таблиц измерений и таблицы фактов

In [3]:
def write_and_read(df, table_name):
    df.coalesce(1).write.jdbc(pg_url, table_name, mode="append", properties=pg_properties)
    return spark.read.jdbc(pg_url, table_name, properties=pg_properties)


g_all = raw_df.selectExpr("customer_country as country", "customer_postal_code as postal_code").withColumn("city", F.lit(None).cast("string")).withColumn("state", F.lit(None).cast("string")) \
    .union(raw_df.selectExpr("store_country as country", "customer_postal_code as postal_code").withColumn("city", F.lit("Unknown").cast("string")).withColumn("state", F.lit(None).cast("string"))) \
    .distinct()
dim_geography = write_and_read(g_all, "dim_geography")


dim_categories = write_and_read(raw_df.select("product_category", "pet_category").distinct(), "dim_categories")


geo_simple = dim_geography.dropDuplicates(["country"])


dim_suppliers_df = raw_df.alias("m") \
    .join(broadcast(geo_simple).alias("g"), F.col("m.supplier_country") == F.col("g.country"), "left") \
    .selectExpr("supplier_name", "supplier_contact as contact_name", "geo_id").distinct().dropDuplicates(["supplier_name"])
dim_suppliers = write_and_read(dim_suppliers_df, "dim_suppliers")


dim_stores_df = raw_df.alias("m") \
    .join(broadcast(geo_simple).alias("g"), F.col("m.store_country") == F.col("g.country"), "left") \
    .selectExpr("store_name", "geo_id").distinct().dropDuplicates(["store_name"])
dim_stores = write_and_read(dim_stores_df, "dim_stores")


dim_customers_df = raw_df.alias("m") \
    .join(broadcast(geo_simple).alias("g"), F.col("m.customer_country") == F.col("g.country"), "left") \
    .selectExpr("customer_email as email", "customer_first_name as first_name", "customer_last_name as last_name", "geo_id").distinct().dropDuplicates(["email"])
dim_customers = write_and_read(dim_customers_df, "dim_customers")


dim_sellers_df = raw_df.alias("m") \
    .join(broadcast(geo_simple).alias("g"), F.col("m.seller_country") == F.col("g.country"), "left") \
    .selectExpr("seller_email as email", "seller_first_name as first_name", "seller_last_name as last_name", "geo_id").distinct().dropDuplicates(["email"])
dim_sellers = write_and_read(dim_sellers_df, "dim_sellers")


dim_products_df = raw_df.alias("m") \
    .join(broadcast(dim_categories).alias("cat"), ["product_category", "pet_category"], "left") \
    .join(broadcast(dim_suppliers).alias("sup"), "supplier_name", "left") \
    .selectExpr("m.sale_product_id as original_product_id", "m.product_name", "m.product_brand as brand", "m.product_price as price", "cat.category_id", "sup.supplier_id") \
    .distinct().dropDuplicates(["original_product_id"])
dim_products = write_and_read(dim_products_df, "dim_products")


fact_sales_df = raw_df.alias("m") \
    .join(broadcast(dim_customers).alias("c"), F.col("m.customer_email") == F.col("c.email")) \
    .join(broadcast(dim_sellers).alias("sel"), F.col("m.seller_email") == F.col("sel.email")) \
    .join(broadcast(dim_products).alias("p"), F.col("m.sale_product_id") == F.col("p.original_product_id")) \
    .join(broadcast(dim_stores).alias("st"), F.col("m.store_name") == F.col("st.store_name")) \
    .selectExpr(
        "m.sale_date",
        "c.customer_id",
        "sel.seller_id",
        "p.product_id",
        "st.store_id",
        "m.sale_quantity as quantity",
        "m.sale_total_price as total_price"
    )
fact_sales_df.coalesce(1).write.jdbc(pg_url, "fact_sales", mode="append", properties=pg_properties)

# Формирование отчётов ClickHouse

In [4]:
def save_to_ch(df, table_name):
    df.na.fill("N/A").na.fill(0).coalesce(1).write.jdbc(ch_url, table_name, mode="overwrite", properties=ch_properties)

f = fact_sales_df.alias("f")
p = dim_products.alias("p")
cat = dim_categories.alias("cat")
c = dim_customers.select("customer_id", "first_name", "last_name", "email", F.col("geo_id").alias("cust_geo_id")).alias("c")
st = dim_stores.select("store_id", "store_name", F.col("geo_id").alias("store_geo_id")).alias("st")
sup = dim_suppliers.select("supplier_id", "supplier_name", F.col("geo_id").alias("sup_geo_id")).alias("sup")
geo = dim_geography.alias("geo")

analytics_base = f.join(p, "product_id") \
    .join(cat, "category_id") \
    .join(c, "customer_id") \
    .join(st, "store_id") \
    .join(sup, "supplier_id")

### 1. Витрина продаж по продуктам Цель: Анализ выручки, количества продаж и популярности продуктов. 
- Топ-10 самых продаваемых продуктов.
- Общая выручка по категориям продуктов.
- Средний рейтинг и количество отзывов для каждого продукта.

In [5]:
save_to_ch(analytics_base.groupBy("product_name", "brand").agg(F.sum("quantity").alias("total_sold")).orderBy(F.desc("total_sold")).limit(10), "v1_1_top_10_selling_products")
save_to_ch(analytics_base.groupBy("product_category", "pet_category").agg(F.sum("total_price").alias("revenue")), "v1_2_revenue_by_category")
save_to_ch(raw_df.groupBy("product_name", "product_brand").agg(F.avg("product_rating").alias("avg_rating"), F.sum("product_reviews").alias("reviews_count")), "v1_3_product_rating_reviews")

### 2. Витрина продаж по клиентам Цель: Анализ покупательского поведения и сегментация клиентов. 
- Топ-10 клиентов с наибольшей общей суммой покупок.
- Распределение клиентов по странам.
- Средний чек для каждого клиента.

In [6]:
v2_base = analytics_base.join(geo, F.col("c.cust_geo_id") == F.col("geo.geo_id"), "left")
save_to_ch(v2_base.groupBy("customer_id", "first_name", "last_name").agg(F.sum("total_price").alias("total_spent")).orderBy(F.desc("total_spent")).limit(10), "v2_1_top_10_customers")
save_to_ch(v2_base.groupBy("country").agg(F.countDistinct("customer_id").alias("client_count")), "v2_2_customers_by_country")
save_to_ch(v2_base.groupBy("customer_id").agg(F.avg("total_price").alias("avg_check")), "v2_3_customer_avg_check")

### 3. Витрина продаж по времени Цель: Анализ сезонности и трендов продаж.
- Месячные и годовые тренды продаж.
- Сравнение выручки за разные периоды (кварталы).
- Средний размер заказа по месяцам.

In [7]:
v3_base = analytics_base.withColumn("year", F.year("sale_date")) \
                        .withColumn("quarter", F.quarter("sale_date")) \
                        .withColumn("month", F.month("sale_date"))
save_to_ch(v3_base.groupBy("year", "month").agg(F.sum("total_price").alias("revenue")).orderBy("year", "month"), "v3_1_monthly_trends")
save_to_ch(v3_base.groupBy("quarter").agg(F.sum("total_price").alias("quarterly_revenue")).orderBy("quarter"), "v3_2_quarterly_comparison")
save_to_ch(v3_base.groupBy("month").agg(F.avg("total_price").alias("avg_order_size")), "v3_3_avg_order_by_month")

### 4. Витрина продаж по магазинам Цель: Анализ эффективности магазинов.
- Топ-5 магазинов с наибольшей выручкой.
- Распределение продаж по городам и странам.
- Средний чек для каждого магазина.

In [8]:
v4_base = analytics_base.join(geo, F.col("st.store_geo_id") == F.col("geo.geo_id"), "left")
save_to_ch(v4_base.groupBy("store_name").agg(F.sum("total_price").alias("revenue")).orderBy(F.desc("revenue")).limit(5), "v4_1_top_5_stores")
save_to_ch(v4_base.groupBy("country", "city").agg(F.sum("total_price").alias("revenue")), "v4_2_store_sales_geo")
save_to_ch(v4_base.groupBy("store_name").agg(F.avg("total_price").alias("avg_check")), "v4_3_store_avg_check")

### 5. Витрина продаж по поставщикам Цель: Анализ эффективности поставщиков.
- Топ-5 поставщиков с наибольшей выручкой.
- Средняя цена товаров от каждого поставщика.
- Распределение продаж по странам поставщиков.

In [9]:
v5_base = analytics_base.join(geo, F.col("sup.sup_geo_id") == F.col("geo.geo_id"), "left")
save_to_ch(v5_base.groupBy("supplier_name").agg(F.sum("total_price").alias("revenue")).orderBy(F.desc("revenue")).limit(5), "v5_1_top_5_suppliers")
save_to_ch(v5_base.groupBy("supplier_name").agg(F.avg("price").alias("avg_unit_price")), "v5_2_supplier_avg_price")
save_to_ch(v5_base.groupBy("country").agg(F.sum("total_price").alias("revenue")), "v5_3_supplier_sales_by_country")

### 6. Витрина качества продукции Цель: Анализ отзывов и рейтингов товаров.
- Продукты с наивысшим и наименьшим рейтингом.
- Корреляция между рейтингом и объемом продаж.
- Продукты с наибольшим количеством отзывов.

In [10]:
v6_data = raw_df.groupBy("product_name", "product_brand").agg(
    F.avg("product_rating").alias("rating"),
    F.sum("product_reviews").alias("reviews"),
    F.sum("sale_total_price").alias("sales_vol")
)
save_to_ch(v6_data.orderBy(F.desc("rating")).limit(5).union(v6_data.orderBy(F.asc("rating")).limit(5)), "v6_1_high_low_rated_products")
save_to_ch(v6_data.select("product_name", "product_brand", "rating", "sales_vol"), "v6_2_rating_vs_sales_vol")
save_to_ch(v6_data.orderBy(F.desc("reviews")).limit(10), "v6_3_most_reviewed_products")